In [1]:
# %% [0] Config + pure helpers
"""
v4 = YOUR BENCHMARK, reproduced exactly, then made less noisy.

The benchmark's recipe (kept bit-for-bit where it matters):
    same photo loading/scaling, same centre crop, same DoG+FFT features, same frozen EfficientNetV2-S,
    same 4-hidden-layer head, same exact-EMD loss, same augmentation distribution (D4 flips/rotations + contrast),
    same sequential batching, same Adam(1e-3), batch 16, steps_per_epoch = n//16, <=25 epochs,
    EarlyStopping(monitor='loss', patience=5, restore_best_weights=True), same SAM stone redraw.

Why it can be improved without touching the recipe (the calculation):
    A trained head f_s(x) is a random curve (seed, batch order, dropout, augmentation). Write the metric
    EMD(y, f) = sum_j w_j |y_j - f_j|, which is CONVEX in f. For S independent members and their mean curve f_bar,
    Jensen gives   EMD(y, f_bar) <= (1/S) sum_s EMD(y, f_s)   for every soil y, so the ensemble can never be worse
    than the AVERAGE single run, and the gap grows with member disagreement. Your 0.30 is one draw from that
    distribution; the ensemble targets its mean (and removes the lottery).
    Test-time augmentation over the 8 symmetries of the centre crop uses exactly the same argument and only inputs
    the head was trained on (no new image regions, unlike earlier versions of mine).
    The pointwise MEDIAN is the L1-optimal point prediction of an empirical predictive distribution and keeps
    monotonicity; it is compared with the mean in CV and used only if clearly better.
    A shrink toward the training-mean curve (lambda) is also decided by CV, never by hand.

What is written: submission.csv (ensemble + stones), sub_control_single.csv (ONE benchmark-style member, no TTA,
+ stones: if it scores ~ your benchmark the cached reproduction is faithful), sub_a20gate.csv, sub_nostones.csv.
"""
import re, io, json, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize

SEED = 0
np.random.seed(SEED)

# ---- switches -------------------------------------------------------------------------------------
USE_SAM, RUN_CV, FORCE_RERUN_SAM = True, True, False
CNN_WEIGHTS = "imagenet"
N_SEEDS = 20                      # ensemble size (heads train in seconds on cached features)
MIXED_BATCHING = True             # half the members use the benchmark's sequential batches, half shuffled
CV_SEEDS = 2                      # CV: members per batching mode
M_AUG = 16                        # cached random augmentations per training photo
CROP_PX, BATCH, LR, EPOCHS, PATIENCE = 400, 16, 1e-3, 25, 5
HEAD_UNITS, DROPOUT = (2048, 512, 256, 128), 0.1
STONE_DEF, GATE_A63, K_CLIP, MIN_FINES, A20_GATE = "bbox", 0.02, (0.8, 1.6), 1.0, 0.25
TARGET_PPM = TRAIN_LIKE_PPM = 4.6
SAM_PPM, SAM_BORDER, SAM_FULLRES_PPM = 2.0, 0.12, 12.0
CAM_EDGE, CAM_A52 = "Motorola Edge", "Samsung A52"
DOG_SIGMAS_MM = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
FFT_BANDS_MM = [(0.5, 0.7), (0.7, 1.4), (1.4, 2.8), (2.8, 5.6), (5.6, 11.2)]

SIZES = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200])
SIZE_COLS = ["0.002", "0.0063", "0.02", "0.063", "0.2", "0.63", "2", "6.3", "20", "63", "200"]
W_EMD = np.diff(np.log10(SIZES))
I20, I63 = SIZE_COLS.index("20"), SIZE_COLS.index("63")
D4 = [(k, f) for f in (False, True) for k in range(4)]          # the 8 symmetries of a square crop


def emd(y_true, y_pred):
    a = np.asarray(y_true, float)[..., :10]
    b = np.asarray(y_pred, float)[..., :10]
    return (np.abs(a - b) * W_EMD).sum(-1)


def postprocess(curve):
    c = np.maximum.accumulate(np.clip(np.asarray(curve, float), 0, 100), axis=-1)
    c[..., -1] = 100.0
    return c


def stable_softplus(x):
    x = np.asarray(x, float)
    return np.maximum(x, 0) + np.log1p(np.exp(-np.abs(x)))


def weibull_curve(lnb, c, d=SIZES):
    lnb = np.asarray(lnb, float)[..., None]
    c = np.maximum(np.asarray(c, float), 0.02)[..., None]
    return 100.0 * (1.0 - np.exp(-np.exp(np.clip(c * (np.log(d) - lnb), -40, 40))))


def make_sites(ids, gap=12):
    parsed = []
    for s in ids:
        m = re.match(r"^([A-Za-z]+)(\d+)$", s)
        parsed.append((m.group(1), int(m.group(2)), s) if m else (s, -10**6, s))
    parsed.sort()
    site, last, cur = {}, None, 0
    for letter, num, s in parsed:
        if last is None or letter != last[0] or num - last[1] > gap:
            cur += 1
        site[s] = f"site{cur:02d}"
        last = (letter, num)
    return pd.Series(site)


def aggregate_soil(curves, sids):
    """Rows (photo x TTA view) -> one curve per soil, INDEXED BY sample_id (ids never travel separately)."""
    df = pd.DataFrame(np.asarray(curves, float), columns=SIZE_COLS)
    df["sid"] = np.asarray(sids)
    out = df.groupby("sid", sort=True).mean()
    return pd.DataFrame(postprocess(out.to_numpy()), index=out.index, columns=SIZE_COLS)


def dihedral(c, k, flip):
    c = np.rot90(c, k)
    return np.ascontiguousarray(c[:, ::-1] if flip else c)


def tf_like_contrast(c, factor):
    """Same maths as tf.image.random_contrast: per-channel mean is the pivot."""
    m = c.mean(axis=(0, 1), keepdims=True)
    return (c - m) * factor + m


def d50_of(curve):
    return float(np.exp(np.interp(50.0, np.asarray(curve, float), np.log(SIZES))))


def stone_correct(curve, a63, a20, k):
    c = np.asarray(curve, float).copy()
    new20 = min(max(100.0 - k * 100.0 * a20, MIN_FINES), 100.0)
    new63 = min(max(100.0 - k * 100.0 * a63, new20), 100.0)
    scale = new20 / max(c[I20], 1e-6)
    if scale < 1:
        c[: I20 + 1] *= scale
    c[I20], c[I63], c[-1] = new20, new63, 100.0
    return postprocess(c)


def stone_correct_a20_only(curve, a20, k):
    c = np.asarray(curve, float).copy()
    new20 = min(max(100.0 - k * 100.0 * a20, MIN_FINES), 100.0)
    scale = new20 / max(c[I20], 1e-6)
    if scale < 1:
        c[: I20 + 1] *= scale
    c[I20] = new20
    c[I63] = max(c[I63], new20)
    c[-1] = 100.0
    return postprocess(c)


def check_submission(df, sample_ids):
    assert list(df.index) == list(sample_ids), "row order / ids differ from sample_submission"
    v = df[SIZE_COLS].to_numpy()
    assert not np.isnan(v).any()
    assert ((v >= 0) & (v <= 100)).all() and (np.diff(v, axis=1) >= -1e-9).all() and (v[:, -1] == 100).all()


# %% [1] Index photos (identical to the benchmark)
import cv2
from PIL import Image, ImageOps

INPUT_DIR = sorted(Path("/kaggle/input").glob("**/Training_labels_updated.csv"))[0].parent
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

CAMERA_RE = re.compile(r"^(Motorola_Edge_60_fusion|Motorola_Edge|Samsung_A52|iPhone_?14|iPhone_?16)_(.+?)(?:_\d+| \(\d+\))?$")
CAMERA_KEY = {"Motorola_Edge_60_fusion": "Motorola Edge 60 Fusion", "Motorola_Edge": CAM_EDGE,
              "Samsung_A52": CAM_A52, "iPhone14": "iPhone 14", "iPhone_14": "iPhone 14",
              "iPhone16": "iPhone 16", "iPhone_16": "iPhone 16"}


def normalize_id(raw):
    s = unicodedata.normalize("NFC", raw).replace("ü", "ue").replace("ö", "oe").replace("ä", "ae")
    return s.replace(",", "_")


labels = pd.read_csv(INPUT_DIR / "Training_labels_updated.csv", dtype={"sample_id": str}).set_index("sample_id")
sample_sub = pd.read_csv(INPUT_DIR / "sample_submission.csv", dtype={"sample_id": str})
ppm_tab = pd.read_csv(INPUT_DIR / "ppm_updated.csv").set_index("phone")

rows = []
for split, pattern in [("train", "Training-All_Photos*"), ("test", "Test_All_Photos*")]:
    for p in sorted(INPUT_DIR.glob(f"{pattern}/**/*")):
        if p.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        m = CAMERA_RE.match(unicodedata.normalize("NFC", p.stem))
        if m is None:
            raise ValueError(f"unexpected file name: {p.name}")
        with Image.open(p) as im:
            w, h = im.size
        rows.append({"split": split, "path": p, "camera": CAMERA_KEY[m.group(1)],
                     "sample_id": normalize_id(m.group(2)), "w": w, "h": h})
photos = pd.DataFrame(rows)
photos["ppm_eff"] = [ppm_tab.loc[c, "ppm"] * max(w, h) / ppm_tab.loc[c, "width"]
                     for c, w, h in zip(photos.camera, photos.w, photos.h)]
assert sorted(set(photos.query("split=='train'").sample_id)) == sorted(labels.index)
assert set(photos.query("split=='test'").sample_id) == set(sample_sub.sample_id)
SID, CAMS = photos.sample_id.to_numpy(), photos.camera.to_numpy()
tr_all = np.where(photos.split.to_numpy() == "train")[0]          # file order, like the benchmark's `tr`
te_all = np.where(photos.split.to_numpy() == "test")[0]
SITE = make_sites(list(labels.index))
print(photos.groupby(["split", "camera"]).agg(photos=("path", "size"), px_per_mm=("ppm_eff", "median")).round(1))
print("sites:", SITE.nunique())

# %% [2] Preprocessing = the benchmark's `load_all`, split into reusable pieces
_HANN = {}


def load_scaled(path, ppm_eff):
    a = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert("RGB"))
    if ppm_eff > 6.0:                                             # sharp (test) photo -> degrade toward train condition
        f = TRAIN_LIKE_PPM / ppm_eff
        a = cv2.resize(a, (round(a.shape[1] * f), round(a.shape[0] * f)), interpolation=cv2.INTER_AREA)
        ppm_eff = TRAIN_LIKE_PPM
    f = TARGET_PPM / ppm_eff
    a = cv2.resize(a, (round(a.shape[1] * f), round(a.shape[0] * f)), interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]:
        a = np.rot90(a)
    return a


def centre_crop(a):
    H, W = a.shape[:2]
    s = min(CROP_PX, H, W); y, x = (H - s) // 2, (W - s) // 2
    return cv2.resize(a[y:y + s, x:x + s], (CROP_PX, CROP_PX), interpolation=cv2.INTER_AREA).astype(np.float32)


def hand_features(a):
    H, W = a.shape[:2]
    c = int(0.25 * min(H, W))
    gray = cv2.cvtColor(a[c:H - c, c:W - c], cv2.COLOR_RGB2GRAY).astype(np.float32)
    blurs = [gray] + [cv2.GaussianBlur(gray, (0, 0), s * TARGET_PPM) for s in DOG_SIGMAS_MM]
    dog = np.array([np.abs(blurs[i] - blurs[i + 1]).mean() for i in range(len(DOG_SIGMAS_MM))], dtype=np.float32)
    n = min(gray.shape); g = gray[:n, :n]
    if n not in _HANN:
        _HANN[n] = np.outer(np.hanning(n), np.hanning(n)).astype(np.float32)
    gc = (g - g.mean()) / (g.std() + 1e-6)
    P = np.abs(np.fft.fftshift(np.fft.fft2(gc * _HANN[n]))) ** 2
    yx = np.indices((n, n)) - n // 2
    fq = np.sqrt(yx[0] ** 2 + yx[1] ** 2) / n
    wl = np.divide(1.0, fq * TARGET_PPM, out=np.full((n, n), np.inf), where=fq > 0)
    fft = np.array([np.log(P[(wl >= lo) & (wl < hi)].mean() + 1e-6) if ((wl >= lo) & (wl < hi)).any() else 0.0
                    for lo, hi in FFT_BANDS_MM], dtype=np.float32)
    return np.concatenate([dog, fft])


# %% [3] One pass: hand features + frozen-EfficientNet embeddings of (a) the 8 symmetries of the centre crop (TTA/eval)
#        and (b) M_AUG random benchmark-style augmentations of every TRAINING crop
import tensorflow as tf


def extract_all():
    tf.keras.utils.set_random_seed(SEED)
    net = tf.keras.applications.EfficientNetV2S(include_top=False, weights=CNN_WEIGHTS, pooling="avg")
    net.trainable = False
    is_train = photos.split.to_numpy() == "train"
    n = len(photos)
    hand, tta, aug = [], None, None
    for i, r in enumerate(photos.itertuples()):
        a = load_scaled(r.path, r.ppm_eff)
        hand.append(hand_features(a))
        crop = centre_crop(a)
        crops = [dihedral(crop, k, f) for k, f in D4]
        if is_train[i]:
            rng = np.random.default_rng(SEED * 100003 + i)
            for _ in range(M_AUG):                               # same distribution as flips + rot90 + random_contrast
                c = dihedral(crop, int(rng.integers(0, 4)), bool(rng.random() < 0.5))
                crops.append(tf_like_contrast(c, rng.uniform(0.85, 1.15)))
        e = net.predict(np.stack(crops).astype("float32"), batch_size=32, verbose=0)
        if tta is None:
            tta = np.zeros((n, 8, e.shape[1]), np.float32); aug = np.zeros((n, M_AUG, e.shape[1]), np.float32)
        tta[i] = e[:8]
        if is_train[i]:
            aug[i] = e[8:]
    tf.keras.backend.clear_session()
    return tta, aug, np.stack(hand).astype(np.float32)


EMB_TTA, EMB_AUG, HAND = extract_all()
print("embeddings: tta", EMB_TTA.shape, "| aug", EMB_AUG.shape, "| hand", HAND.shape)


# %% [4] The benchmark head and its exact training loop, on cached features
def emd_loss_tf(y_true, y_pred):
    d = tf.constant(SIZES[:10], tf.float32); w = tf.constant(W_EMD, tf.float32)
    ln_b, c = y_pred[:, 0], tf.nn.softplus(y_pred[:, 1]) + 0.02
    F = 100.0 * (1.0 - tf.exp(-tf.pow(d[None, :] / tf.exp(ln_b)[:, None], c[:, None])))
    return tf.reduce_sum(tf.abs(F - y_true) * w[None, :], axis=1)


def build_head(d_in, seed):
    tf.keras.utils.set_random_seed(seed)
    inp = tf.keras.Input((d_in,)); h = inp
    for n in HEAD_UNITS:
        h = tf.keras.layers.Dense(n, activation="relu")(h)
        h = tf.keras.layers.Dropout(DROPOUT)(h)
    model = tf.keras.Model(inp, tf.keras.layers.Dense(2, activation="linear")(h))
    model.compile(optimizer=tf.keras.optimizers.Adam(LR), loss=emd_loss_tf)
    return model


def fit_member(tr_ph, seed, sequential):
    """One benchmark-style training run. Returns (model, hand_stats). tr_ph is in file order."""
    stats = (HAND[tr_ph].mean(0), HAND[tr_ph].std(0) + 1e-6)          # scaler fitted on the training photos only
    hs = ((HAND[tr_ph] - stats[0]) / stats[1]).astype(np.float32)
    Y = labels.loc[SID[tr_ph], SIZE_COLS[:10]].to_numpy("float32")
    D = EMB_AUG.shape[2] + HAND.shape[1]
    rng = np.random.default_rng(seed)
    n = len(tr_ph)

    def gen():
        pos = 0
        while True:
            if sequential:                                           # benchmark: consecutive photos, cycling
                idx = (pos + np.arange(BATCH)) % n; pos = (pos + BATCH) % n
            else:
                idx = rng.integers(0, n, BATCH)
            v = rng.integers(0, M_AUG, BATCH)
            yield np.concatenate([EMB_AUG[tr_ph[idx], v], hs[idx]], 1).astype("float32"), Y[idx]

    ds = tf.data.Dataset.from_generator(gen, output_signature=(tf.TensorSpec((BATCH, D), tf.float32),
                                                               tf.TensorSpec((BATCH, 10), tf.float32)))
    model = build_head(D, seed)
    model.fit(ds, epochs=EPOCHS, steps_per_epoch=max(1, n // BATCH), verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="loss", patience=PATIENCE, restore_best_weights=True)])
    return model, stats


def predict_soil(model, stats, te_ph, tta=True):
    """Soil-indexed curves for the photos in te_ph. tta=False = identity view only (benchmark inference)."""
    views = list(range(8)) if tta else [0]
    X = np.concatenate([EMB_TTA[te_ph][:, views].reshape(-1, EMB_TTA.shape[2]),
                        np.repeat((HAND[te_ph] - stats[0]) / stats[1], len(views), axis=0)], 1).astype("float32")
    z = np.asarray(model(X, training=False))
    curves = weibull_curve(z[:, 0], stable_softplus(z[:, 1]) + 0.02)
    return aggregate_soil(curves, np.repeat(SID[te_ph], len(views)))


def combine(members, how="mean"):
    arr = np.stack([m.to_numpy() for m in members])
    out = arr.mean(0) if how == "mean" else np.median(arr, 0)
    return pd.DataFrame(postprocess(out), index=members[0].index, columns=SIZE_COLS)


def shrink(df, prior, lam):
    return pd.DataFrame(postprocess((1 - lam) * df.to_numpy() + lam * np.asarray(prior)[None, :]), index=df.index, columns=SIZE_COLS)


def soil_emd(df):
    return pd.Series(emd(labels.loc[df.index, SIZE_COLS].to_numpy(float), df.to_numpy()), index=df.index)


# %% [5] Validation with the SAME recipe: site hold-out and site x phone-swap (cheap, because features are cached)
def cv_jobs(scheme):
    for site in sorted(SITE.unique()):
        held = np.isin(SID, list(SITE.index[SITE == site]))
        if scheme == "site":
            yield "all", tr_all[~held[tr_all]], tr_all[held[tr_all]]
        else:
            for cx, cy in [(CAM_EDGE, CAM_A52), (CAM_A52, CAM_EDGE)]:
                a, b = tr_all[(~held[tr_all]) & (CAMS[tr_all] == cx)], tr_all[held[tr_all] & (CAMS[tr_all] == cy)]
                if len(a) and len(b):
                    yield f"{cx[:3]}>{cy[:3]}", a, b


LAMBDAS = (0.0, 0.1, 0.2)
AGG, LAM = "mean", 0.0
if RUN_CV:
    rec = {}
    for scheme in ("site", "xphone"):
        for tag, a, b in cv_jobs(scheme):
            prior = labels.loc[sorted(set(SID[a])), SIZE_COLS].mean().to_numpy()
            mem = {True: [], False: []}
            ctrl = None
            for sd in range(CV_SEEDS):
                for seq in (True, False):
                    model, stats = fit_member(a, 1000 + sd, seq)
                    mem[seq].append(predict_soil(model, stats, b, tta=True))
                    if ctrl is None and seq:
                        ctrl = predict_soil(model, stats, b, tta=False)            # = benchmark-style single run
                    tf.keras.backend.clear_session()
            allm = mem[True] + mem[False]
            variants = {"single(no TTA)": ctrl, "single+TTA(seq)": mem[True][0], "single+TTA(shuf)": mem[False][0],
                        "ens seq only": combine(mem[True]), "ens shuf only": combine(mem[False]),
                        "ens mean": combine(allm), "ens median": combine(allm, "median")}
            for lam in LAMBDAS[1:]:
                variants[f"ens mean + shrink {lam}"] = shrink(combine(allm), prior, lam)
            for name, df in variants.items():
                for s, e in soil_emd(df).items():
                    rec.setdefault((name, scheme), {})[f"{s}|{tag}"] = e
    rows_out = {}
    for (name, scheme), d in rec.items():
        ser = pd.Series(d)
        rows_out.setdefault(name, {})[scheme] = ser.groupby(ser.index.str.split("|").str[0]).mean().mean()
    cv = pd.DataFrame(rows_out).T
    cv["combined"] = 0.5 * cv["site"] + 0.5 * cv["xphone"]
    print("\nCross-validated EMD of the benchmark recipe and its ensembles (lower is better)")
    print(cv.round(2).to_string())
    AGG = "median" if cv.loc["ens median", "combined"] < cv.loc["ens mean", "combined"] - 0.3 else "mean"
    best = {lam: cv.loc["ens mean" if lam == 0 else f"ens mean + shrink {lam}", "combined"] for lam in LAMBDAS}
    cand = min(best, key=best.get)
    LAM = cand if best[0.0] - best[cand] >= 0.3 else 0.0
    print(f"chosen aggregation: {AGG} | shrink lambda: {LAM}")
    print("CAUTION: CV rehearses Motorola<->Samsung only. The iPhone/field shift cannot be validated with these labels.")

# %% [6] Final ensemble on all training photos, TTA on the test photos
prior_all = labels.mean().to_numpy()
members, ctrl_single = [], None
for s in range(N_SEEDS):
    seq = (s % 2 == 0) if MIXED_BATCHING else True
    model, stats = fit_member(tr_all, SEED + s, seq)
    members.append(predict_soil(model, stats, te_all, tta=True))
    if s == 0:
        ctrl_single = predict_soil(model, stats, te_all, tta=False)           # benchmark-style single run, centre crop only
    tf.keras.backend.clear_session()
assert all(list(m.index) == list(members[0].index) for m in members)
ens = shrink(combine(members, AGG), prior_all, LAM)
member_spread = pd.Series({s: float(np.mean([emd(m.loc[s].to_numpy(), ens.loc[s].to_numpy()) for m in members])) for s in ens.index})
print(f"ensemble of {N_SEEDS} heads | test soils:", list(ens.index))

# %% [7] Stones with SAM (same rule as the benchmark; cached) + calibration on H374
CACHE = OUTPUT_DIR / "stone_cache.csv"


def stone_geometry(m):
    ys, xs = np.nonzero(m)
    bbox = max(ys.max() - ys.min(), xs.max() - xs.min()) / SAM_PPM
    ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([xs, ys]))))[::-1]
    return {"bbox": bbox, "pca_major": 4 * np.sqrt(max(ev[0], 0)) / SAM_PPM,
            "pca_minor": 4 * np.sqrt(max(ev[1], 0)) / SAM_PPM, "equiv": 2 * np.sqrt(m.sum() / np.pi) / SAM_PPM}


def stones_from_masks(masks, tot):
    if not masks:
        return []
    order = sorted(range(len(masks)), key=lambda i: -int(masks[i].sum()))
    taken, out = np.zeros_like(masks[0], bool), []
    for i in order:
        m = masks[i]; sz = int(m.sum())
        if sz < (8.0 * SAM_PPM) ** 2 or sz > 0.5 * tot or (m & taken).sum() > 0.5 * sz:
            continue
        g = stone_geometry(m)
        if sz / (np.pi * (g["bbox"] * SAM_PPM / 2) ** 2 + 1e-6) < 0.35:
            continue
        taken |= m
        out.append({"area_frac": sz / tot, **g})
    return out


def load_work(path, ppm):
    a = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert("RGB"))
    a = cv2.resize(a, None, fx=SAM_PPM / ppm, fy=SAM_PPM / ppm, interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]:
        a = np.rot90(a)
    H, W = a.shape[:2]; c = SAM_BORDER
    return np.ascontiguousarray(a[int(H * c):int(H * (1 - c)), int(W * c):int(W * (1 - c))])


def measure_stones():
    import torch
    from transformers import pipeline
    gen = pipeline("mask-generation", model="facebook/sam-vit-base", device=0 if torch.cuda.is_available() else -1)
    rec = []
    for r in photos[photos.ppm_eff >= SAM_FULLRES_PPM].itertuples():
        rgb = load_work(r.path, r.ppm_eff)
        out = gen(Image.fromarray(rgb), points_per_batch=64, pred_iou_thresh=0.86, stability_score_thresh=0.9)
        st = stones_from_masks([np.asarray(m).astype(bool) for m in out["masks"]], rgb.shape[0] * rgb.shape[1])
        row = {"sample_id": r.sample_id, "n": len(st), "cover": sum(s["area_frac"] for s in st)}
        for d in ("bbox", "pca_major", "pca_minor", "equiv"):
            row[f"A63_{d}"] = sum(s["area_frac"] for s in st if s[d] > 63)
            row[f"A20_{d}"] = sum(s["area_frac"] for s in st if s[d] > 20)
        rec.append(row)
    return pd.DataFrame(rec)


stone_ms, K_CAL = None, 1.0
if USE_SAM:
    try:
        meas = pd.read_csv(CACHE, dtype={"sample_id": str}) if (CACHE.exists() and not FORCE_RERUN_SAM) else measure_stones()
        meas.to_csv(CACHE, index=False)
        stone_ms = meas.groupby("sample_id").mean(numeric_only=True)
        a20_cal = float(stone_ms.loc["H374", f"A20_{STONE_DEF}"]) if "H374" in stone_ms.index else 0.0
        if a20_cal > 0.0:
            K_CAL = float(np.clip((100.0 - labels.loc["H374", "20"]) / (100.0 * a20_cal), *K_CLIP))
        else:
            warnings.warn("H374 calibration unusable: using Delesse k = 1")
        print(f"stones: def={STONE_DEF} k={K_CAL:.3f} (clip {K_CLIP}) | H374 A20={a20_cal:.3f}")
        print(stone_ms.loc[[s for s in stone_ms.index if s in set(sample_sub.sample_id)],
                           ["n", "cover", f"A63_{STONE_DEF}", f"A20_{STONE_DEF}"]].round(3))
    except Exception as e:
        warnings.warn(f"SAM stage disabled: {e}")
        USE_SAM = False


def with_stones(curves_df, soft_a20=False):
    if not USE_SAM or stone_ms is None:
        return curves_df
    out = {}
    for sid in curves_df.index:
        a63, a20 = ((float(stone_ms.loc[sid, f"A63_{STONE_DEF}"]), float(stone_ms.loc[sid, f"A20_{STONE_DEF}"]))
                    if sid in stone_ms.index else (0.0, 0.0))
        cur = curves_df.loc[sid].to_numpy()
        if a63 >= GATE_A63:
            out[sid] = stone_correct(cur, a63, a20, K_CAL)
        elif soft_a20 and a20 >= A20_GATE:
            out[sid] = stone_correct_a20_only(cur, a20, K_CAL)
        else:
            out[sid] = cur
    return pd.DataFrame(out, index=SIZE_COLS).T


# %% [8] Submissions + diagnostics
ids = list(sample_sub.sample_id)
variants = {"submission": with_stones(ens), "sub_control_single": with_stones(ctrl_single),
            "sub_a20gate": with_stones(ens, soft_a20=True), "sub_nostones": ens}
for name, df in variants.items():
    df = pd.DataFrame(postprocess(df.loc[ids, SIZE_COLS].to_numpy()), index=ids, columns=SIZE_COLS)
    check_submission(df, ids)
    df.reset_index().rename(columns={"index": "sample_id"}).to_csv(OUTPUT_DIR / f"{name}.csv", index=False, float_format="%.4f")
prim = pd.read_csv(OUTPUT_DIR / "submission.csv", dtype={"sample_id": str}).set_index("sample_id")
diag = pd.DataFrame({"n_photos": photos[photos.split == "test"].groupby("sample_id").size(),
                     "member_spread": member_spread.round(1),
                     "d50_mm": pd.Series({s: round(d50_of(prim.loc[s]), 3) for s in prim.index}),
                     "<0.063mm_%": prim["0.063"].round(1), "<2mm_%": prim["2"].round(1), "<63mm_%": prim["63"].round(1)})
print(diag.loc[ids].to_string())
json.dump({"n_seeds": N_SEEDS, "aggregation": AGG, "shrink_lambda": LAM, "k": K_CAL, "stone_def": STONE_DEF},
          open(OUTPUT_DIR / "run_summary.json", "w"), indent=2)
print("written:", sorted(p.name for p in OUTPUT_DIR.glob("sub*.csv")))

                               photos  px_per_mm
split camera                                    
test  iPhone 14                    14       13.9
      iPhone 16                    21       19.5
train Motorola Edge                69        4.6
      Motorola Edge 60 Fusion       3       12.5
      Samsung A52                  55        4.6
sites: 13


I0000 00:00:1793362913.245021      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793362913.248689      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


2026-10-30 12:22:15.855026: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 12:22:16.016514: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 12:22:17.123193: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 12:22:17.293385: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 12:22:18.888165: E external/local_xla/xla/stream_

embeddings: tta (162, 8, 1280) | aug (162, 16, 1280) | hand (162, 11)

Cross-validated EMD of the benchmark recipe and its ensembles (lower is better)
                        site  xphone  combined
single(no TTA)         40.26   45.02     42.64
single+TTA(seq)        39.97   44.52     42.25
single+TTA(shuf)       47.77   48.66     48.22
ens seq only           44.85   44.74     44.79
ens shuf only          44.42   44.80     44.61
ens mean               44.42   44.17     44.30
ens median             44.91   44.05     44.48
ens mean + shrink 0.1  44.52   45.07     44.80
ens mean + shrink 0.2  47.89   48.42     48.16
chosen aggregation: mean | shrink lambda: 0.0
CAUTION: CV rehearses Motorola<->Samsung only. The iPhone/field shift cannot be validated with these labels.
ensemble of 20 heads | test soils: ['HPC_Airbus BS10-4bis7', 'HPC_Airbus BS12-5bis8', 'HPC_Airbus BS6-3', 'HPC_Audorfring', 'HPC_Kleinkummerfeld 18-3', 'HPC_Kleinkummerfeld 2-2', 'HPC_Kleinkummerfeld 2-3', 'HPC_Kleinkummerfe

config.json:   0%|          | 0.00/6.57k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


stones: def=bbox k=1.305 (clip (0.8, 1.6)) | H374 A20=0.180
                                n  cover  A63_bbox  A20_bbox
sample_id                                                   
HPC_Airbus BS10-4bis7       7.333  0.020     0.000     0.004
HPC_Airbus BS12-5bis8       2.667  0.008     0.000     0.000
HPC_Airbus BS6-3            3.000  0.009     0.000     0.000
HPC_Audorfring             36.800  0.157     0.000     0.048
HPC_Kleinkummerfeld 18-3    0.333  0.001     0.000     0.000
HPC_Kleinkummerfeld 2-2     2.333  0.006     0.000     0.000
HPC_Kleinkummerfeld 2-3     1.333  0.005     0.000     0.000
HPC_Kleinkummerfeld 9-4     1.000  0.003     0.000     0.002
HPC_Muenster_BS6_9_0-10m   64.400  0.838     0.267     0.724
HPC_Testfeld Lidl WHV     104.250  0.635     0.000     0.407
                          n_photos  member_spread  d50_mm  <0.063mm_%  <2mm_%  <63mm_%
HPC_Airbus BS6-3                 3           23.6   0.419        17.0    82.2     99.8
HPC_Audorfring                   5